# 03e — Pre-Flight Feature Engineering (Problem C)

Build the final feature matrix for the pre-flight risk model.

**Input:** `data/processed/preflight_weather_enriched.parquet`  
**Output:** `data/processed/preflight_features_final.parquet`

In [ ]:
import os, sys
from pathlib import Path

root = Path.cwd()
while not (root / 'pyproject.toml').exists():
    root = root.parent
os.chdir(root)

# Ensure src/ is on the Python path for local imports
src_path = str(root / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print(f'Working directory: {root}')
print(f'sys.path[0]: {sys.path[0]}')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from features.preflight import add_temporal_features, compute_historical_rates, categorize_weather

IN_PATH  = Path('data/processed/preflight_weather_enriched.parquet')
OUT_PATH = Path('data/processed/preflight_features_final.parquet')

df = pd.read_parquet(IN_PATH)
print(f'Loaded dataset: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
print(f'Incident rate: {df["incident"].mean()*100:.2f}%')

## 1. Temporal Features

In [ ]:
df = add_temporal_features(df)

# Route ID for historical rate tracking
df['route_id'] = df['ORIGIN'].astype(str) + '_' + df['DEST'].astype(str)

print('Temporal features added:')
print(df[['month_sin', 'month_cos', 'dow_sin', 'dow_cos', 'hour']].describe())

## 2. Weather Features

In [ ]:
df = categorize_weather(df)

# Impute missing weather with global medians
weather_cols = ['temp', 'dwpt', 'rhum', 'prcp', 'wspd', 'pres']
for col in weather_cols:
    if col in df.columns:
        med = df[col].median()
        n_missing = df[col].isna().sum()
        df[col] = df[col].fillna(med)
        if n_missing > 0:
            print(f'  Imputed {n_missing:,} missing {col} values with median={med:.2f}')
    else:
        df[col] = 0.0
        print(f'  WARNING: {col} not found, set to 0')

print('Weather features ready.')

## 3. Historical Incident Rates (Leakage-Free)

In [ ]:
# compute_historical_rates uses expanding().mean().shift(1) — no data leakage
print('Computing historical incident rates...')

df['airport_risk_rate']  = compute_historical_rates(df, 'ORIGIN')
df['carrier_risk_rate']  = compute_historical_rates(df, 'OP_UNIQUE_CARRIER')
df['route_risk_rate']    = compute_historical_rates(df, 'route_id')

# Fill initial nulls (no prior history) with 0
for col in ['airport_risk_rate', 'carrier_risk_rate', 'route_risk_rate']:
    df[col] = df[col].fillna(0)

print('Historical rates computed:')
print(df[['airport_risk_rate', 'carrier_risk_rate', 'route_risk_rate']].describe())

## 4. Feature Correlation Analysis

In [ ]:
features = [
    'month_sin', 'month_cos', 'dow_sin', 'dow_cos', 'hour',
    'temp', 'rhum', 'prcp', 'wspd', 'pres',
    'airport_risk_rate', 'carrier_risk_rate', 'route_risk_rate',
    'DISTANCE'
]

# Only include features that exist in the DataFrame
available_features = [f for f in features if f in df.columns]
print(f'Features available: {available_features}')

fig, ax = plt.subplots(figsize=(12, 10))
corr_matrix = df[available_features + ['incident']].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', ax=ax)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 5. Save

In [ ]:
# Save the enriched feature matrix
df.to_parquet(OUT_PATH, index=False)
print(f'Saved final feature matrix to {OUT_PATH}')
print(f'Shape: {df.shape}')
print(f'Features saved: {available_features}')